<div dir="rtl">
<h1>اصلاح کوچک روی وزن ثابت</h1>
<p>درس 74 از 76 · برای تنظیم رفتار، لازم است همهٔ وزن‌ها تغییر کنند؟ · <code dir="ltr">65b-lora</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-10/chapter-02/65b-lora.html">📖 بازگشت به همین درس</a></p>
<p>شاخهٔ LoRA را بنویسید و ببینید کدام Parameter در گام اول Gradient می‌گیرد.</p><p>پیش‌نیاز: ضرب ماتریسی، Linear، Parameter ثابت و قابل آموزش.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>اگر A و B هر دو صفر باشند، آیا این شاخه از صفر حرکت می‌کند؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import torch
from torch import nn
from mini_gpt.config import ModelConfig
from mini_gpt.model import MiniGPT
torch.set_num_threads(1)
torch.manual_seed(12)
project_model = MiniGPT(ModelConfig(8,4,12,3,1,0.0))
W = project_model.language_model_head.weight.detach().clone()
A = nn.Parameter(torch.randn(2,12)*0.01)
B = nn.Parameter(torch.zeros(8,2))
x = torch.randn(3,12)
print('base weight shape:',tuple(W.shape),'adapter scalars:',A.numel()+B.numel())
print('base output shape:',tuple((x@W.T).shape))

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>lora_forward(x,W,A,B,alpha) خروجی xWᵀ + (alpha/r)(xAᵀ)Bᵀ را بدهد؛ r تعداد سطرهای A است. W شکل (M,C)، A شکل (r,C)، B شکل (M,r) دارد. هیچ Tensor را درجا تغییر ندهید.</p>
</div>

In [ ]:
def lora_forward(x, W, A, B, alpha):
    # TODO: شاخهٔ کم‌رتبه به خروجی پایه اضافه می‌شود
    return None

In [ ]:
def test_exercise():
    result = lora_forward(x,W,A,B,2.0)
    if result is None:
        return False
    torch.testing.assert_close(result,x@W.T)
    trial_b = torch.ones_like(B)
    torch.testing.assert_close(lora_forward(x,W,A,trial_b,4.0),x@(W+2.0*(trial_b@A)).T)
    target = torch.ones_like(result)
    ((result-target)**2).mean().backward()
    assert W.grad is None
    assert A.grad is not None and torch.count_nonzero(A.grad)==0
    assert B.grad is not None and B.grad.norm()>0
    assert A.numel()+B.numel()==40
    return True
exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: lora_forward')

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط Rank را عوض کنید و Parameterهای شاخه را بشمارید. کاهش Rank به‌تنهایی کیفیت Fine-Tuning را پیش‌بینی نمی‌کند.</p>
</div>

In [ ]:
for rank in (1,2,4):
    a = torch.zeros(rank,12)
    b = torch.zeros(8,rank)
    print(rank,'adapter:',a.numel()+b.numel(),'base still needed:',W.numel())

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>صفرکردن هر دو عامل، Gradient هرکدام را در عامل صفر دیگر ضرب می‌کند. initialize_adapter(out_features,in_features,Rank) دو nn.Parameter بسازد: A با مقدار تصادفی کوچک و B صفر.</p>
</div>

In [ ]:
wrong_a = nn.Parameter(torch.zeros(2,12))
wrong_b = nn.Parameter(torch.zeros(8,2))
((x@W.T+(x@wrong_a.T)@wrong_b.T-1)**2).mean().backward()
print('both-zero gradient norms:',wrong_a.grad.norm().item(),wrong_b.grad.norm().item())

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def initialize_adapter(out_features, in_features, rank):
    # TODO: خروجی آغازین پایه بماند، اما شاخه بتواند یاد بگیرد
    return None

In [ ]:
def test_repair():
    result = initialize_adapter(8,12,2)
    if result is None:
        return False
    a,b = result
    assert isinstance(a,nn.Parameter) and isinstance(b,nn.Parameter)
    assert a.shape==(2,12) and b.shape==(8,2)
    assert torch.count_nonzero(a)>0 and torch.count_nonzero(b)==0
    a2,b2 = initialize_adapter(3,5,1)
    assert a2.shape==(1,5) and b2.shape==(3,1)
    assert a2.requires_grad and b2.requires_grad
    return True
repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: initialize_adapter')

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>W یک کپی ثابت از سر خروجی MiniGPT است؛ شاخهٔ آموزشی به خود مدل نصب نشده است. LoRA روش انتخاب Parameterهای قابل آموزش است و می‌تواند کنار هدف SFT به کار رود؛ هدف آموزشی جداگانه‌ای نیست.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>چرا ۴۰ Parameter قابل آموزش به معنی ۴۰ Parameter لازم برای اجرای مدل نیست؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-10/chapter-02/65b-lora.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/65b-lora.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>